# 03 — AQE Join Optimization

Adaptive Query Execution behavior for joins in Spark.

## 00 — Setup

```text
SparkSession
    │
    ├── warehouse: /tmp/spark-warehouse
    ├── shuffle partitions: 32
    ├── AQE toggled per section
    └── join plans inspected with explain()
```

In [1]:
from pyspark.sql import SparkSession, functions as F
import time

spark = (
    SparkSession.builder
    .appName("aqe_join_optimization")
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse")
    .getOrCreate()
)

spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

spark


## 01 — Sample data

```text
fact_orders
  order_id
  customer_id ─────────────┐
  product_id               │
  amount                   │
                           ├── join key
customers                  │
  customer_id ─────────────┘
  country
  segment
```

In [2]:
fact_orders = (
    spark.range(0, 2_000_000)
    .withColumnRenamed("id", "order_id")
    .withColumn("customer_id", (F.col("order_id") % 200_000).cast("long"))
    .withColumn("product_id", (F.col("order_id") % 25_000).cast("long"))
    .withColumn("amount", ((F.col("order_id") % 500) + 10).cast("double"))
)

customers = (
    spark.range(0, 200_000)
    .withColumnRenamed("id", "customer_id")
    .withColumn("country", F.expr("CASE WHEN customer_id % 5 = 0 THEN 'SK' WHEN customer_id % 5 = 1 THEN 'CZ' WHEN customer_id % 5 = 2 THEN 'DE' WHEN customer_id % 5 = 3 THEN 'AT' ELSE 'PL' END"))
    .withColumn("segment", F.expr("CASE WHEN customer_id % 4 = 0 THEN 'enterprise' WHEN customer_id % 4 = 1 THEN 'smb' WHEN customer_id % 4 = 2 THEN 'consumer' ELSE 'public' END"))
)

fact_orders.count(), customers.count()


(2000000, 200000)

## 02 — Baseline without AQE

```text
logical join
    │
    ├── planning happens before runtime statistics
    └── physical plan fixed before execution

AQE disabled
```

In [3]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

baseline_join = fact_orders.join(customers, on="customer_id", how="inner")
baseline_join.explain(mode="formatted")


== Physical Plan ==
* Project (11)
+- * SortMergeJoin Inner (10)
   :- * Sort (5)
   :  +- Exchange (4)
   :     +- * Project (3)
   :        +- * Filter (2)
   :           +- * Range (1)
   +- * Sort (9)
      +- Exchange (8)
         +- * Project (7)
            +- * Range (6)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project [codegen id : 1]
Output [4]: [id#0L AS order_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 25000) AS product_id#3L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: hashpartitioning(customer_id#2L, 32), ENSURE_REQUIREMENTS, [plan_id=101]

(5) Sort [codegen id : 2]
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: [customer_id#2L ASC NULLS FIRST], false, 0

(6) Range [co

## 03 — AQE enabled

```text
logical join
    │
    ├── initial physical plan
    ├── runtime shuffle statistics
    └── adaptive final physical plan

AQE enabled
```

In [4]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

aqe_join = fact_orders.join(customers, on="customer_id", how="inner")
aqe_join.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project
Output [4]: [id#0L AS order_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 25000) AS product_id#3L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: hashpartitioning(customer_id#2L, 32), ENSURE_REQUIREMENTS, [plan_id=158]

(5) Sort
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: [customer_id#2L ASC NULLS FIRST], false, 0

(6) Range
Output [1]: [id#5L]
Arguments: Ra

## 04 — Runtime plan after action

```text
AQE plan before action
        │
        ├── count()
        ▼
AQE final plan with runtime decisions
```

In [5]:
rows = aqe_join.count()
print(rows)
aqe_join.explain(mode="formatted")


[Stage 10:>                                                         (0 + 2) / 2]

2000000
== Physical Plan ==
AdaptiveSparkPlan (12)
+- Project (11)
   +- SortMergeJoin Inner (10)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (9)
         +- Exchange (8)
            +- Project (7)
               +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project
Output [4]: [id#0L AS order_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 25000) AS product_id#3L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: hashpartitioning(customer_id#2L, 32), ENSURE_REQUIREMENTS, [plan_id=158]

(5) Sort
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: [customer_id#2L ASC NULLS FIRST], false, 0

(6) Range
Output [1]: [id#5L]
Argum

## 05 — Dynamic broadcast opportunity

```text
large-looking source
        │
        ├── filter at runtime
        ▼
small relation after filtering
        │
        └── AQE may convert shuffle join to broadcast join
```

In [6]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.autoBroadcastJoinThreshold", 20 * 1024 * 1024)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

filtered_customers = customers.filter(F.col("country") == "SK")
dynamic_join = fact_orders.join(filtered_customers, on="customer_id", how="inner")

dynamic_join.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (10)
         +- Exchange (9)
            +- Project (8)
               +- Filter (7)
                  +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : (CASE WHEN (((id#0L % 200000) % 5) = 0) THEN true WHEN (((id#0L % 200000) % 5) = 1) THEN false WHEN (((id#0L % 200000) % 5) = 2) THEN false WHEN (((id#0L % 200000) % 5) = 3) THEN false ELSE false END AND isnotnull((id#0L % 200000)))

(3) Project
Output [4]: [id#0L AS order_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 25000) AS product_id#3L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: hashparti

In [7]:
dynamic_join.count()
dynamic_join.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (13)
+- Project (12)
   +- SortMergeJoin Inner (11)
      :- Sort (5)
      :  +- Exchange (4)
      :     +- Project (3)
      :        +- Filter (2)
      :           +- Range (1)
      +- Sort (10)
         +- Exchange (9)
            +- Project (8)
               +- Filter (7)
                  +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : (CASE WHEN (((id#0L % 200000) % 5) = 0) THEN true WHEN (((id#0L % 200000) % 5) = 1) THEN false WHEN (((id#0L % 200000) % 5) = 2) THEN false WHEN (((id#0L % 200000) % 5) = 3) THEN false ELSE false END AND isnotnull((id#0L % 200000)))

(3) Project
Output [4]: [id#0L AS order_id#1L, (id#0L % 200000) AS customer_id#2L, (id#0L % 25000) AS product_id#3L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [4]: [order_id#1L, customer_id#2L, product_id#3L, amount#4]
Arguments: hashparti

## 06 — Coalescing shuffle partitions

```text
many small shuffle partitions
        │
        ├── AQE reads runtime sizes
        └── combines small partitions

less scheduling overhead
```

In [8]:
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")

coalesce_demo = (
    fact_orders
    .join(customers, "customer_id")
    .groupBy("country", "segment")
    .agg(F.sum("amount").alias("revenue"))
)

coalesce_demo.explain(mode="formatted")
coalesce_demo.collect()
coalesce_demo.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (15)
+- HashAggregate (14)
   +- Exchange (13)
      +- HashAggregate (12)
         +- Project (11)
            +- SortMergeJoin Inner (10)
               :- Sort (5)
               :  +- Exchange (4)
               :     +- Project (3)
               :        +- Filter (2)
               :           +- Range (1)
               +- Sort (9)
                  +- Exchange (8)
                     +- Project (7)
                        +- Range (6)


(1) Range
Output [1]: [id#0L]
Arguments: Range (0, 2000000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#0L]
Condition : isnotnull((id#0L % 200000))

(3) Project
Output [2]: [(id#0L % 200000) AS customer_id#2L, cast(((id#0L % 500) + 10) as double) AS amount#4]
Input [1]: [id#0L]

(4) Exchange
Input [2]: [customer_id#2L, amount#4]
Arguments: hashpartitioning(customer_id#2L, 128), ENSURE_REQUIREMENTS, [plan_id=632]

(5) Sort
Input [2]: [customer_id#2L, amount#4]
Arguments: [customer_id#2L ASC NULLS FIR

== Physical Plan ==
AdaptiveSparkPlan (31)
+- == Final Plan ==
   ResultQueryStage (21)
   +- * HashAggregate (20)
      +- AQEShuffleRead (19)
         +- ShuffleQueryStage (18), Statistics(sizeInBytes=2000.0 B, rowCount=40)
            +- Exchange (17)
               +- * HashAggregate (16)
                  +- * Project (15)
                     +- * BroadcastHashJoin Inner BuildRight (14)
                        :- AQEShuffleRead (6)
                        :  +- ShuffleQueryStage (5), Statistics(sizeInBytes=45.8 MiB, rowCount=2.00E+6)
                        :     +- Exchange (4)
                        :        +- * Project (3)
                        :           +- * Filter (2)
                        :              +- * Range (1)
                        +- BroadcastQueryStage (13), Statistics(sizeInBytes=17.5 MiB, rowCount=2.00E+5)
                           +- BroadcastExchange (12)
                              +- AQEShuffleRead (11)
                                 +- Shuffl

## 07 — Skewed join data

```text
normal keys      evenly distributed
hot key = 1      many more rows
        │
        ▼
one reducer receives too much data
```

In [9]:
normal_events = (
    spark.range(0, 500_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", ((F.col("event_id") % 199_999) + 2).cast("long"))
)

hot_events = (
    spark.range(0, 1_500_000)
    .withColumnRenamed("id", "event_id")
    .withColumn("customer_id", F.lit(1).cast("long"))
)

skewed_events = normal_events.unionByName(hot_events)

skewed_events.groupBy("customer_id").count().orderBy(F.desc("count")).show(5)


+-----------+-------+
|customer_id|  count|
+-----------+-------+
|          1|1500000|
|        224|      3|
|         93|      3|
|        364|      3|
|        171|      3|
+-----------+-------+
only showing top 5 rows


## 08 — Skewed join without AQE

```text
skewed_events ── shuffle(customer_id) ──┐
                                       ├── join
customers     ── shuffle(customer_id) ─┘

hot key creates one heavy shuffle partition
```

In [10]:
spark.conf.set("spark.sql.adaptive.enabled", "false")
spark.conf.set("spark.sql.shuffle.partitions", "32")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

skew_no_aqe = skewed_events.join(customers, "customer_id")
skew_no_aqe.explain(mode="formatted")


== Physical Plan ==
* Project (14)
+- * SortMergeJoin Inner (13)
   :- * Sort (8)
   :  +- Exchange (7)
   :     +- Union (6)
   :        :- * Project (3)
   :        :  +- * Filter (2)
   :        :     +- * Range (1)
   :        +- * Project (5)
   :           +- * Range (4)
   +- * Sort (12)
      +- Exchange (11)
         +- * Project (10)
            +- * Range (9)


(1) Range [codegen id : 1]
Output [1]: [id#59L]
Arguments: Range (0, 500000, step=1, splits=Some(2))

(2) Filter [codegen id : 1]
Input [1]: [id#59L]
Condition : isnotnull(((id#59L % 199999) + 2))

(3) Project [codegen id : 1]
Output [2]: [id#59L AS event_id#60L, ((id#59L % 199999) + 2) AS customer_id#61L]
Input [1]: [id#59L]

(4) Range [codegen id : 2]
Output [1]: [id#62L]
Arguments: Range (0, 1500000, step=1, splits=Some(2))

(5) Project [codegen id : 2]
Output [2]: [id#62L AS event_id#63L, 1 AS customer_id#64L]
Input [1]: [id#62L]

(6) Union

(7) Exchange
Input [2]: [event_id#60L, customer_id#61L]
Arguments: hashpa

## 09 — Skewed join with AQE skew handling

```text
AQE detects large shuffle partition
        │
        ├── split skewed partition
        ├── replicate matching side as needed
        └── process hot key in smaller chunks
```

In [11]:
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "2")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "1024")
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

skew_with_aqe = skewed_events.join(customers, "customer_id")
skew_with_aqe.count()
skew_with_aqe.explain(mode="formatted")


== Physical Plan ==
AdaptiveSparkPlan (15)
+- Project (14)
   +- SortMergeJoin Inner (13)
      :- Sort (8)
      :  +- Exchange (7)
      :     +- Union (6)
      :        :- Project (3)
      :        :  +- Filter (2)
      :        :     +- Range (1)
      :        +- Project (5)
      :           +- Range (4)
      +- Sort (12)
         +- Exchange (11)
            +- Project (10)
               +- Range (9)


(1) Range
Output [1]: [id#59L]
Arguments: Range (0, 500000, step=1, splits=Some(2))

(2) Filter
Input [1]: [id#59L]
Condition : isnotnull(((id#59L % 199999) + 2))

(3) Project
Output [2]: [id#59L AS event_id#60L, ((id#59L % 199999) + 2) AS customer_id#61L]
Input [1]: [id#59L]

(4) Range
Output [1]: [id#62L]
Arguments: Range (0, 1500000, step=1, splits=Some(2))

(5) Project
Output [2]: [id#62L AS event_id#63L, 1 AS customer_id#64L]
Input [1]: [id#62L]

(6) Union

(7) Exchange
Input [2]: [event_id#60L, customer_id#61L]
Arguments: hashpartitioning(customer_id#61L, 32), ENSURE_RE

## 10 — AQE timing comparison

```text
same join
    │
    ├── AQE disabled → fixed shuffle plan
    └── AQE enabled  → runtime optimized plan
```

In [12]:
def measure(label, enabled):
    spark.conf.set("spark.sql.adaptive.enabled", "true" if enabled else "false")
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    df = skewed_events.join(customers, "customer_id")
    start = time.time()
    rows = df.count()
    seconds = round(time.time() - start, 3)
    return (label, rows, seconds)

timing_rows = [
    measure("aqe_disabled", False),
    measure("aqe_enabled", True),
]

spark.createDataFrame(timing_rows, ["case", "rows", "seconds"]).show(truncate=False)


+------------+-------+-------+
|case        |rows   |seconds|
+------------+-------+-------+
|aqe_disabled|1999998|1.136  |
|aqe_enabled |1999998|0.64   |
+------------+-------+-------+



## 11 — Decision summary

```text
AQE helps most when runtime data differs from compile-time assumptions
        │
        ├── unexpected small side      → dynamic broadcast
        ├── too many small partitions  → coalesce shuffle partitions
        └── skewed shuffle partitions  → skew join splitting
```

In [13]:
summary = [
    ("dynamic broadcast", "filtered or underestimated small side", "AQE may switch join strategy at runtime"),
    ("shuffle coalescing", "many small shuffle partitions", "AQE reduces partition count after seeing sizes"),
    ("skew join handling", "hot keys and uneven reducers", "AQE splits large shuffle partitions"),
    ("not a silver bullet", "bad join key or massive broadcast side", "modeling and partitioning still matter"),
]

spark.createDataFrame(summary, ["feature", "when_it_matters", "effect"]).show(truncate=False)


+-------------------+--------------------------------------+----------------------------------------------+
|feature            |when_it_matters                       |effect                                        |
+-------------------+--------------------------------------+----------------------------------------------+
|dynamic broadcast  |filtered or underestimated small side |AQE may switch join strategy at runtime       |
|shuffle coalescing |many small shuffle partitions         |AQE reduces partition count after seeing sizes|
|skew join handling |hot keys and uneven reducers          |AQE splits large shuffle partitions           |
|not a silver bullet|bad join key or massive broadcast side|modeling and partitioning still matter        |
+-------------------+--------------------------------------+----------------------------------------------+

